In [1]:
import numpy as np
import random
import pandas as pd

# Set up global parameters for consistency
M = 5  # Number of tasks (m)
N = 4  # Number of nodes (n)
SEED = 42

# ------------------------------------------------------------
# Helper Functions: State Index Mapping
# ------------------------------------------------------------
def state_index(i, j, n_nodes):
    """Map (task i, node j) -> linear state index."""
    return i * n_nodes + j

def state_from_index(s, n_nodes):
    """Inverse mapping from linear index to (i, j)."""
    return s // n_nodes, s % n_nodes


# ------------------------------------------------------------
# Environment / MDP model for QTO (QTOEnvMDP)
# ------------------------------------------------------------
class QTOEnvMDP:
    """
    Models the Quantum Task Offloading (QTO) environment and MDP dynamics.
    Parameters and models are based on the QTOP paper.
    """
    def __init__(self, m, n, phi=0.85, gamma=0.95, seed=0):
        
        random.seed(seed)
        np.random.seed(seed)

        self.m = m
        self.n = n
        self.phi = phi
        self.gamma = gamma

        # --- Task parameters (ϕ_q, ϕ_d, ϕ_s, ϕ_T, ϕ_P) ---
        self.tasks = []
        for _ in range(m):
            self.tasks.append({
                "qubits": random.choice([2, 4, 8]),       # ϕ_g^q: required qubits
                "depth": random.uniform(10, 50),          # ϕ_g^d: circuit depth
                "shots": random.choice([100, 500, 1000]), # ϕ_g^s: number of shots
                "deadline": random.uniform(5.0, 30.0),    # ϕ_g^T: threshold time
                "max_price": random.uniform(5.0, 20.0)    # ϕ_g^P: maximum allowable price
            })

        # --- Node parameters (n_q, n_s, p_j, ρ_j) ---
        self.nodes = []
        for _ in range(n):
            self.nodes.append({
                "qubits": random.choice([4, 8, 16, 32]),  # n_j^q: available qubits
                "clops": random.uniform(1e3, 5e3),        # n_j^s: CLOPS
                "power": random.uniform(0.1, 0.5),        # p_j: power per qubit
                "price": random.uniform(0.1, 1.0)         # ρ_j: price per second
            })

        self.base_node_load = np.zeros(n)  # Used for VI approximation

        # Precompute global maxima for normalization: δ_m, e_m, P_m, σ_m
        self.delta_max, self.energy_max, self.price_max, self.load_max = self._precompute_global_maxima()

    # --------------------------------------------------------
    # Delay model (Eq. 3, 5)
    # --------------------------------------------------------
    def computation_delay(self, task_i, node_k):
        """δ_c_gj = (ϕ_d_g × ϕ_s_g) / n_s_j (Eq. 3)"""
        depth = task_i["depth"]
        shots = task_i["shots"]
        clops = node_k["clops"]
        return (depth * shots) / max(clops, 1e-9) 

    def waiting_delay(self, node_k, extra_depth=0.0):
        """Approximation of waiting delay for VI reward."""
        total_depth = self.base_node_load[node_k] + extra_depth
        clops = self.nodes[node_k]["clops"]
        return (total_depth * 1.0) / max(clops, 1e-9)  

    def total_delay(self, i, k):
        """δ_g = δ_c_gj + δ_w_g (Eq. 5)"""
        task = self.tasks[i]
        comp = self.computation_delay(task, self.nodes[k])
        wait = self.waiting_delay(k, extra_depth=task["depth"])
        return comp + wait

    # --------------------------------------------------------
    # Energy model (Eq. 9)
    # --------------------------------------------------------
    def energy_consumption(self, i, k):
        """e_c_g = δ_c_gj × ϕ_q_g × p_j (Eq. 9)"""
        task = self.tasks[i]
        node = self.nodes[k]
        δc = self.computation_delay(task, node)
        return δc * task["qubits"] * node["power"]

    # --------------------------------------------------------
    # Price model (Eq. 14)
    # --------------------------------------------------------
    def price_consumption(self, i, k):
        """P_g = ρ_j × δ_c_gj (Eq. 14)"""
        task = self.tasks[i]
        node = self.nodes[k]
        δc = self.computation_delay(task, node)
        return node["price"] * δc

    # --------------------------------------------------------
    # Load model (Variance based on σ_j) (Eq. 12)
    # --------------------------------------------------------
    def load_variance_if_assign(self, i, k):
        """Approximation of Var(σ) for MDP reward."""
        loads = self.base_node_load.copy()
        loads[k] += self.tasks[i]["depth"] 
        mean_load = np.mean(loads)
        return np.mean((loads - mean_load) ** 2)

    # --------------------------------------------------------
    # Feasibility and Transition Probability
    # --------------------------------------------------------
    def feasible_nodes(self, i):
        """Returns list of nodes satisfying n_j^q >= ϕ_g^q (Eq. 24)."""
        req_q = self.tasks[i]["qubits"]
        return [k for k in range(self.n) if self.nodes[k]["qubits"] >= req_q]

    def P_qubit(self, i, k):
        """Checks qubit constraint (Eq. 24). Returns 1 if feasible, 0 otherwise."""
        req_q = self.tasks[i]["qubits"]
        node_q = self.nodes[k]["qubits"]
        return 1 if node_q >= req_q else 0

    def P_node(self, i, a, k):
        """Transition probability for node selection based on action 'a' and node success ϕ."""
        F = self.feasible_nodes(i)
        if k not in F:
            return 0.0

        if len(F) <= 1:
            return 1.0 if k == a else 0.0

        # P_node(s_{i,k} | a) logic from the QRL problem text
        if k == a:
            return self.phi
        else:
            return (1 - self.phi) / (len(F) - 1)

    def transition_prob(self, s, a, s_next):
        """Full transition probability P(s' | s, a)."""
        i, j = state_from_index(s, self.n)
        i2, k = state_from_index(s_next, self.n)
        if i != i2:
            return 0.0 

        # P(s_{i,k} | s_{i,j}, a=k) = P_node(i, a, k) * P_qubit(i, k)
        return self.P_node(i, a, k) * self.P_qubit(i, k)

    # --------------------------------------------------------
    # Normalized Reward (R = sum of (1 - Normalized Cost))
    # --------------------------------------------------------
    def normalized_reward(self, i, k, feasible=True,
                          wD=0.4, wE=0.2, wC=0.2, wB=0.2, penalty=100.0):
        """
        Calculates the normalized reward R, aiming to maximize utility (minimize cost).
        """
        if not feasible:
            return -penalty

        D = self.total_delay(i, k)
        E = self.energy_consumption(i, k)
        C = self.price_consumption(i, k)
        B = self.load_variance_if_assign(i, k)

        # Maxima (δ_m, e_m, P_m, σ_m) for normalization
        δm = max(self.delta_max, 1e-9)
        em = max(self.energy_max, 1e-9)
        Pm = max(self.price_max, 1e-9)
        σm = max(self.load_max, 1e-9)

        # Calculate normalized utility (1 - Normalized Cost)
        Dn = 1 - (D / δm)
        En = 1 - (E / em)
        Cn = 1 - (C / Pm)
        Bn = 1 - (B / σm)

        # Ensure normalized metrics are within [0, 1]
        Dn = max(0.0, min(1.0, Dn))
        En = max(0.0, min(1.0, En))
        Cn = max(0.0, min(1.0, Cn))
        Bn = max(0.0, min(1.0, Bn))

        R = wD * Dn + wE * En + wC * Cn + wB * Bn
        return R

    def _precompute_global_maxima(self):
        """Calculates theoretical maximum costs for normalization (δ_m, e_m, P_m, σ_m)."""
        
        # δ_m (Max delay): all tasks on slowest node (min CLOPS) (Eq. 8 approximation)
        slowest_idx = np.argmin([nd["clops"] for nd in self.nodes])
        slow_node = self.nodes[slowest_idx]
        delta_max = 0.0
        for t in self.tasks:
            delta_max += (t["depth"] * t["shots"]) / max(slow_node["clops"], 1e-9)

        # e_m (Max energy): sum of max possible individual energy costs (Eq. 11 approximation)
        energy_max = sum([
            self.computation_delay(t, self.nodes[np.argmin([nd["clops"] for nd in self.nodes])]) * t["qubits"] * self.nodes[np.argmax([nd["power"] for nd in self.nodes])]["power"]
            for t in self.tasks
        ]) 
        
        # P_m (Max price): sum of max possible individual price costs (Eq. 16 approximation)
        price_max = sum([
             self.nodes[np.argmax([nd["price"] for nd in self.nodes])]["price"] * self.computation_delay(t, self.nodes[np.argmin([nd["clops"] for nd in self.nodes])])
            for t in self.tasks
        ])

        # σ_m (Max load): total sum of all task depths (Eq. 13 approximation)
        load_max = sum(t["depth"] for t in self.tasks)

        return delta_max, energy_max, price_max, load_max


# ------------------------------------------------------------
# Value Iteration Algorithm (Policy Finding)
# ------------------------------------------------------------
def value_iteration(env: QTOEnvMDP, max_iter=300, tol=1e-6):
    """
    Computes the optimal Value function V(s) and Policy π(s) for the MDP.
    """
    S = env.m * env.n
    A = env.n

    V = np.zeros(S)
    policy = np.zeros(S, dtype=int)

    for it in range(max_iter):
        delta = 0.0
        V_new = np.copy(V)

        for s in range(S):
            i, j = state_from_index(s, env.n)

            action_values = []
            for a in range(A):
                expected_val = 0.0

                # Sum over all possible next states s_{i,k}
                for k in range(env.n):
                    s_next = state_index(i, k, env.n)

                    # Transition probability P(s_{i,k} | s_{i,j}, a)
                    P = env.transition_prob(s, a, s_next)
                    if P == 0.0:
                        continue

                    feasible = (env.P_qubit(i, k) == 1)
                    R = env.normalized_reward(i, k, feasible=feasible)

                    # Bellman Expectation Equation: R(s,a) + γ V(s')
                    expected_val += P * (R + env.gamma * V[s_next])

                action_values.append(expected_val)

            # Value and policy update (Max operator)
            best_val = max(action_values)
            best_action = int(np.argmax(action_values))

            V_new[s] = best_val
            policy[s] = best_action

            delta = max(delta, abs(V_new[s] - V[s]))

        V = V_new

        if delta < tol:
            # print(f"Value Iteration converged in {it+1} iterations.")
            break

    return V, policy

# ------------------------------------------------------------
# Sequential QTOP Simulation (Policy Execution)
# ------------------------------------------------------------
def run_qtop_simulation_from_vi_policy(env: QTOEnvMDP, policy):
    """
    Simulates the sequential QTOP decision process using the VI policy, 
    enforcing urgency (Eq. 30) and sequential workload accumulation,
    and returns detailed per-task metrics.
    """
    
    # --- 1. Determine Task Urgency Order (U_g) ---
    
    urgency_values = []
    for i in range(env.m):
        task = env.tasks[i]
        feasible = env.feasible_nodes(i)
        
        if not feasible:
            urgency_values.append(np.inf) # Max urgency
            continue

        # Use the fastest feasible node (max CLOPS) for the initial urgency calculation.
        clops_values = [env.nodes[k]["clops"] for k in feasible]
        fastest_node_idx = feasible[np.argmax(clops_values)]
        
        delta_gc = env.computation_delay(task, env.nodes[fastest_node_idx])
        
        # U_g = 1 / (phi_g^T - delta_gj^c) (Eq. 17)
        phi_g_T = task["deadline"]
        urgency = 1.0 / max(1e-9, (phi_g_T - delta_gc))
        urgency_values.append(urgency)

    # Sort tasks by ascending urgency (Eq. 30: U_{\pi(1)} <= ... <= U_{\pi(M)})
    # Then reverse the indices to get DESCENDING (Highest Urgency first)
    task_order_indices = np.argsort(urgency_values)[::-1] # FIX: Reverse for highest priority first
    
    # --- 2. Sequential Assignment and Workload Update ---
    
    final_assignments = {} 
    current_workload_time = np.zeros(env.n) # total time δ_j
    current_workload_load = np.zeros(env.n) # total depth σ_j
    
    total_metrics = {'delay': 0.0, 'energy': 0.0, 'price': 0.0}
    per_task_results = [] # Stores per-task breakdown
    
    for rank, i in enumerate(task_order_indices): # Iterate over corrected order
        
        # Check for infeasible tasks that ended up with np.inf urgency (shouldn't be in the viable list, but safety check)
        if urgency_values[i] == np.inf:
            continue
            
        task = env.tasks[i]
        
        # Policy lookup: Use policy[s_{i, 0}] 
        s_initial = state_index(i, 0, env.n) 
        best_node_k = policy[s_initial] 
        
        # --- Real-Time Metrics Recalculation ---
        
        # Computation delay (δ_c_gj) (Eq. 3)
        delta_gc = env.computation_delay(task, env.nodes[best_node_k])
        
        # Waiting delay (δ_w_g) is the accumulated workload (δ_j) (Eq. 4)
        delta_gw = current_workload_time[best_node_k]
        
        # Total task time (δ_g) (Eq. 5)
        delta_g = delta_gc + delta_gw
        
        # Price (P_g) (Eq. 14)
        price_g = env.price_consumption(i, best_node_k)
        
        # Energy (e_g^c) (Eq. 9)
        energy_g = env.energy_consumption(i, best_node_k)
        
        # Check Qubit constraint (Eq. 24)
        if env.P_qubit(i, best_node_k) == 0:
             continue
        
        # Check Deadline constraint (Eq. 26)
        if delta_g >= task["deadline"]:
            continue
            
        # Check Price constraint (Eq. 27)
        if price_g >= task["max_price"]:
            continue

        # --- Successful Assignment: Update Workload and Metrics ---
        
        current_workload_time[best_node_k] += delta_gc # Update δ_j
        current_workload_load[best_node_k] += task["depth"] # Update σ_j
        
        # Update system metrics (e^t, P^t, δ^t is max later)
        total_metrics['energy'] += energy_g 
        total_metrics['price'] += price_g
        
        final_assignments[i] = best_node_k
        
        # Store per-task results
        per_task_results.append({
            'Task Index': i,
            'Assigned Node': best_node_k,
            'Queueing Delay (δ_w)': delta_gw,
            'Execution Delay (δ_c)': delta_gc,
            'Total Time (δ_g)': delta_g,
            'Deadline (ϕ_T)': task["deadline"],
            'Energy (e_g^c)': energy_g,
            'Price (P_g)': price_g
        })

    # --- 3. Final System Load and Delay Calculation ---
    
    # δ^t = max(δ_j) (Eq. 7)
    if current_workload_time.size > 0:
        total_metrics['delay'] = np.max(current_workload_time)
    
    mean_load = np.sum(current_workload_load) / env.n # σ̄
    load_variation = np.sum((current_workload_load - mean_load) ** 2) / env.n # Var(σ)
    
    load_max_sq = env.load_max ** 2 
    normalized_load_balance = load_variation / max(load_max_sq, 1e-9)

    total_metrics['load_variation'] = load_variation
    total_metrics['normalized_load_balance'] = normalized_load_balance
    
    return final_assignments, total_metrics, per_task_results

# ------------------------------------------------------------
# Demo Execution
# ------------------------------------------------------------
if __name__ == "__main__":
    
    env = QTOEnvMDP(M, N, phi=0.85, gamma=0.95, seed=SEED)
    
    print("####################################################")
    print("# INITIAL QTOP PARAMETERS (SEED 42) #")
    print("####################################################")

    # 1. Print Initial Parameters
    print("\n--- Initial Quantum Node Parameters (N=4) ---")
    node_params = pd.DataFrame(env.nodes)
    node_params.index.name = 'Node Index'
    print(node_params.to_string(float_format="{:.3f}".format))
        
    print("\n--- Initial Quantum Task Parameters (M=5) ---")
    task_params = pd.DataFrame(env.tasks)
    task_params.index.name = 'Task Index'
    print(task_params.to_string(float_format="{:.3f}".format))

    # 2. Run Policy Finding (Value Iteration)
    print("\nStarting Value Iteration to find Optimal Policy...")
    V, policy = value_iteration(env, max_iter=300, tol=1e-6)

    # 3. Print Optimal Policy
    print("\n\n=== Optimal Policy (π(s_{i,j}) from VI) ===")
    for i in range(M):
        for j in range(N):
            s = state_index(i, j, N)
            print(f"s({i},{j}) -> best action (node) = {policy[s]}")
    
    # 4. Run Sequential Simulation
    print("\n\n####################################################")
    print("# EXECUTING QTOP SEQUENTIAL ASSIGNMENT #")
    print("####################################################")
    
    assignments, system_metrics, per_task_data = run_qtop_simulation_from_vi_policy(env, policy)

    # 5. Output Results
    
    print("\n\n####################################################")
    print("# 📊 PER-TASK PERFORMANCE BREAKDOWN 📊 #")
    print("####################################################")

    if per_task_data:
        df_results = pd.DataFrame(per_task_data)
        df_results = df_results.set_index('Task Index')
        
        # Rename columns for clarity in the output table
        df_results.columns = [
            'Assigned Node', 'Queueing Delay (δ_w) [s]', 'Execution Delay (δ_c) [s]', 
            'Total Time (δ_g) [s]', 'Deadline (ϕ_T) [s]', 
            'Energy (e_g^c) [J]', 'Price (P_g) [$]'
        ]
        
        # Add status based on total time vs deadline
        df_results['Status'] = np.where(df_results['Total Time (δ_g) [s]'] < df_results['Deadline (ϕ_T) [s]'], 'OK', 'FAIL')
        
        print(df_results.to_string(float_format="{:.4f}".format))
    else:
        print("No tasks were successfully offloaded due to initial constraint failures.")

    # 6. Print Aggregated System Metrics
    print("\n\n####################################################")
    print("# 📈 AGGREGATED SYSTEM OBJECTIVES 📈 #")
    print("####################################################")
    df_agg = pd.DataFrame([system_metrics]).T
    df_agg.columns = ['Value']
    print(df_agg.to_string(float_format="{:.4f}".format))

####################################################
# INITIAL QTOP PARAMETERS (SEED 42) #
####################################################

--- Initial Quantum Node Parameters (N=4) ---
            qubits    clops  power  price
Node Index                               
0                8 3792.558  0.236  0.240
1               16 1408.841  0.252  0.423
2               16 3414.904  0.423  0.757
3                4 4892.463  0.251  0.597

--- Initial Quantum Task Parameters (M=5) ---
            qubits  depth  shots  deadline  max_price
Task Index                                           
0                8 14.453   1000    11.876      8.348
1                8 14.100   1000    27.304      6.304
2                4 11.271    100    10.466     12.580
3                2 32.450   1000    21.247     13.174
4                2 27.968    500    25.236      5.097

Starting Value Iteration to find Optimal Policy...


=== Optimal Policy (π(s_{i,j}) from VI) ===
s(0,0) -> best action (node) = 0
s

In [3]:
import numpy as np
import random
import pandas as pd

# Set up global parameters for consistency
M = 5  # Number of tasks (m)
N = 4  # Number of nodes (n)
SEED = 42

# ------------------------------------------------------------
# Helper Functions: State Index Mapping
# ------------------------------------------------------------
def state_index(i, j, n_nodes):
    """Map (task i, node j) -> linear state index."""
    return i * n_nodes + j

def state_from_index(s, n_nodes):
    """Inverse mapping from linear index to (i, j)."""
    return s // n_nodes, s % n_nodes


# ------------------------------------------------------------
# Environment / MDP model for QTO (QTOEnvMDP)
# ------------------------------------------------------------
class QTOEnvMDP:
    """
    Models the Quantum Task Offloading (QTO) environment and MDP dynamics.
    """
    def __init__(self, m, n, phi=0.85, gamma=0.95, seed=0):
        
        random.seed(seed)
        np.random.seed(seed)

        self.m = m
        self.n = n
        self.phi = phi
        self.gamma = gamma

        # --- Task parameters (ϕ_q, ϕ_d, ϕ_s, ϕ_T, ϕ_P) ---
        self.tasks = []
        for _ in range(m):
            self.tasks.append({
                "qubits": random.choice([2, 4, 8]),
                "depth": random.uniform(10, 50),
                "shots": random.choice([100, 500, 1000]),
                "deadline": random.uniform(5.0, 30.0),
                "max_price": random.uniform(5.0, 20.0)
            })

        # --- Node parameters (n_q, n_s, p_j, ρ_j) ---
        self.nodes = []
        for _ in range(n):
            self.nodes.append({
                "qubits": random.choice([4, 8, 16, 32]),
                "clops": random.uniform(1e3, 5e3),
                "power": random.uniform(0.1, 0.5),
                "price": random.uniform(0.1, 1.0)
            })

        self.base_node_load = np.zeros(n)

        # Precompute global maxima for normalization: δ_m, e_m, P_m, σ_m
        self.delta_max, self.energy_max, self.price_max, self.load_max = self._precompute_global_maxima()

    # --------------------------------------------------------
    # Delay model (Eq. 3, 5)
    # --------------------------------------------------------
    def computation_delay(self, task_i, node_k):
        """δ_c_gj = (ϕ_d_g × ϕ_s_g) / n_s_j (Eq. 3)"""
        depth = task_i["depth"]
        shots = task_i["shots"]
        clops = node_k["clops"]
        return (depth * shots) / max(clops, 1e-9) 

    def total_delay(self, i, k):
        """Approximation of total delay for VI (δ_g = δ_c_gj + δ_w_g)."""
        task = self.tasks[i]
        comp = self.computation_delay(task, self.nodes[k])
        # Approximate waiting delay for MDP: (base load + current depth) / CLOPS
        wait = (self.base_node_load[k] + task["depth"]) / max(self.nodes[k]["clops"], 1e-9)
        return comp + wait

    # --------------------------------------------------------
    # Energy, Price, Load, Feasibility, and Transition Logic (omitted for brevity)
    # --------------------------------------------------------

    def energy_consumption(self, i, k):
        """e_c_g = δ_c_gj × ϕ_q_g × p_j (Eq. 9)"""
        task = self.tasks[i]
        node = self.nodes[k]
        δc = self.computation_delay(task, node)
        return δc * task["qubits"] * node["power"]

    def price_consumption(self, i, k):
        """P_g = ρ_j × δ_c_gj (Eq. 14)"""
        task = self.tasks[i]
        node = self.nodes[k]
        δc = self.computation_delay(task, node)
        return node["price"] * δc

    def load_variance_if_assign(self, i, k):
        """Approximation of Var(σ) for MDP reward."""
        loads = self.base_node_load.copy()
        loads[k] += self.tasks[i]["depth"] 
        mean_load = np.mean(loads)
        return np.mean((loads - mean_load) ** 2)

    def feasible_nodes(self, i):
        """Returns list of nodes satisfying n_j^q >= ϕ_g^q (Eq. 24)."""
        req_q = self.tasks[i]["qubits"]
        return [k for k in range(self.n) if self.nodes[k]["qubits"] >= req_q]

    def P_qubit(self, i, k):
        """Checks qubit constraint (Eq. 24). Returns 1 if feasible, 0 otherwise."""
        req_q = self.tasks[i]["qubits"]
        node_q = self.nodes[k]["qubits"]
        return 1 if node_q >= req_q else 0

    def P_node(self, i, a, k):
        """Transition probability for node selection based on action 'a' and node success ϕ."""
        F = self.feasible_nodes(i)
        if k not in F:
            return 0.0

        if len(F) <= 1:
            return 1.0 if k == a else 0.0

        if k == a:
            return self.phi
        else:
            return (1 - self.phi) / (len(F) - 1)

    def transition_prob(self, s, a, s_next):
        """Full transition probability P(s' | s, a)."""
        i, j = state_from_index(s, self.n)
        i2, k = state_from_index(s_next, self.n)
        if i != i2:
            return 0.0 

        return self.P_node(i, a, k) * self.P_qubit(i, k)

    def normalized_reward(self, i, k, feasible=True,
                          wD=0.4, wE=0.2, wC=0.2, wB=0.2, penalty=100.0):
        """
        Calculates the normalized reward R, aiming to maximize utility (minimize cost).
        """
        if not feasible:
            return -penalty

        D = self.total_delay(i, k)
        E = self.energy_consumption(i, k)
        C = self.price_consumption(i, k)
        B = self.load_variance_if_assign(i, k)

        # Maxima (δ_m, e_m, P_m, σ_m) for normalization
        δm = max(self.delta_max, 1e-9)
        em = max(self.energy_max, 1e-9)
        Pm = max(self.price_max, 1e-9)
        σm = max(self.load_max, 1e-9)

        # Calculate normalized utility (1 - Normalized Cost)
        Dn = 1 - (D / δm)
        En = 1 - (E / em)
        Cn = 1 - (C / Pm)
        Bn = 1 - (B / σm)

        # Ensure normalized metrics are within [0, 1]
        Dn = max(0.0, min(1.0, Dn))
        En = max(0.0, min(1.0, En))
        Cn = max(0.0, min(1.0, Cn))
        Bn = max(0.0, min(1.0, Bn))

        R = wD * Dn + wE * En + wC * Cn + wB * Bn
        return R

    def _precompute_global_maxima(self):
        """Calculates theoretical maximum costs for normalization (δ_m, e_m, P_m, σ_m)."""
        
        # δ_m (Max delay): all tasks on slowest node (min CLOPS) (Eq. 8 approximation)
        slowest_idx = np.argmin([nd["clops"] for nd in self.nodes])
        slow_node = self.nodes[slowest_idx]
        delta_max = 0.0
        for t in self.tasks:
            delta_max += (t["depth"] * t["shots"]) / max(slow_node["clops"], 1e-9)

        # e_m (Max energy): sum of max possible individual energy costs (Eq. 11 approximation)
        energy_max = sum([
            self.computation_delay(t, self.nodes[np.argmin([nd["clops"] for nd in self.nodes])]) * t["qubits"] * self.nodes[np.argmax([nd["power"] for nd in self.nodes])]["power"]
            for t in self.tasks
        ]) 
        
        # P_m (Max price): sum of max possible individual price costs (Eq. 16 approximation)
        price_max = sum([
             self.nodes[np.argmax([nd["price"] for nd in self.nodes])]["price"] * self.computation_delay(t, self.nodes[np.argmin([nd["clops"] for nd in self.nodes])])
            for t in self.tasks
        ])

        # σ_m (Max load): total sum of all task depths (Eq. 13 approximation)
        load_max = sum(t["depth"] for t in self.tasks)

        return delta_max, energy_max, price_max, load_max


# ------------------------------------------------------------
# Value Iteration Algorithm (Policy Finding)
# ------------------------------------------------------------
def value_iteration(env: QTOEnvMDP, max_iter=300, tol=1e-6):
    """
    Computes the optimal Value function V(s) and Policy π(s) for the MDP.
    """
    S = env.m * env.n
    A = env.n

    V = np.zeros(S)
    policy = np.zeros(S, dtype=int)

    for it in range(max_iter):
        delta = 0.0
        V_new = np.copy(V)

        for s in range(S):
            i, j = state_from_index(s, env.n)

            action_values = []
            for a in range(A):
                expected_val = 0.0

                # Sum over all possible next states s_{i,k}
                for k in range(env.n):
                    s_next = state_index(i, k, env.n)

                    P = env.transition_prob(s, a, s_next)
                    if P == 0.0:
                        continue

                    feasible = (env.P_qubit(i, k) == 1)
                    R = env.normalized_reward(i, k, feasible=feasible)

                    expected_val += P * (R + env.gamma * V[s_next])

                action_values.append(expected_val)

            best_val = max(action_values)
            best_action = int(np.argmax(action_values))

            V_new[s] = best_val
            policy[s] = best_action

            delta = max(delta, abs(V_new[s] - V[s]))

        V = V_new

        if delta < tol:
            break

    return V, policy

# ------------------------------------------------------------
# Sequential QTOP Simulation (Policy Execution)
# ------------------------------------------------------------
def run_qtop_simulation_from_vi_policy(env: QTOEnvMDP, policy):
    """
    Simulates the sequential QTOP decision process using the VI policy, 
    enforcing urgency (highest U_g first) and sequential workload accumulation.
    """
    
    # --- 1. Determine Task Urgency Order (U_g) ---
    
    urgency_values = []
    for i in range(env.m):
        task = env.tasks[i]
        feasible = env.feasible_nodes(i)
        
        if not feasible:
            urgency_values.append(np.inf)
            continue

        # Use the fastest feasible node (max CLOPS) for the initial urgency calculation.
        clops_values = [env.nodes[k]["clops"] for k in feasible]
        fastest_node_idx = feasible[np.argmax(clops_values)]
        
        delta_gc = (task["depth"] * task["shots"]) / max(env.nodes[fastest_node_idx]["clops"], 1e-9)
        
        # U_g = 1 / (phi_g^T - delta_gj^c) (Eq. 17)
        phi_g_T = task["deadline"]
        urgency = 1.0 / max(1e-9, (phi_g_T - delta_gc))
        urgency_values.append(urgency)

    # Sort tasks by ascending urgency and then reverse the indices for DESCENDING order
    task_order_indices = np.argsort(urgency_values)[::-1] 
    
    # --- 2. Sequential Assignment and Workload Update ---
    
    final_assignments = {} 
    current_workload_time = np.zeros(env.n)
    current_workload_load = np.zeros(env.n)
    
    total_metrics = {'delay': 0.0, 'energy': 0.0, 'price': 0.0}
    per_task_results = []
    
    for rank, i in enumerate(task_order_indices):
        if urgency_values[i] == np.inf:
            continue
            
        task = env.tasks[i]
        
        # Policy lookup
        s_initial = state_index(i, 0, env.n) 
        best_node_k = policy[s_initial] 
        
        # --- Real-Time Metrics Recalculation ---
        
        delta_gc = (task["depth"] * task["shots"]) / max(env.nodes[best_node_k]["clops"], 1e-9)
        delta_gw = current_workload_time[best_node_k]
        delta_g = delta_gc + delta_gw
        
        price_g = env.nodes[best_node_k]["price"] * delta_gc
        energy_g = delta_gc * task["qubits"] * env.nodes[best_node_k]["power"]
        
        # Check Constraints
        if env.P_qubit(i, best_node_k) == 0: continue
        if delta_g >= task["deadline"]: continue
        if price_g >= task["max_price"]: continue

        # --- Successful Assignment: Update Workload and Metrics ---
        
        current_workload_time[best_node_k] += delta_gc
        current_workload_load[best_node_k] += task["depth"]
        
        total_metrics['energy'] += energy_g 
        total_metrics['price'] += price_g
        
        final_assignments[i] = best_node_k
        
        # Store per-task results
        per_task_results.append({
            'Task Index': i,
            'Assigned Node': best_node_k,
            'Queueing Delay (δ_w)': delta_gw,
            'Execution Delay (δ_c)': delta_gc,
            'Total Time (δ_g)': delta_g,
            'Deadline (ϕ_T)': task["deadline"],
            'Energy (e_g^c)': energy_g,
            'Price (P_g)': price_g
        })

    # --- 3. Final System Load and Delay Calculation ---
    
    if current_workload_time.size > 0:
        total_metrics['delay'] = np.max(current_workload_time)
    
    mean_load = np.sum(current_workload_load) / env.n
    load_variation = np.sum((current_workload_load - mean_load) ** 2) / env.n
    
    load_max_sq = env.load_max ** 2 
    normalized_load_balance = load_variation / max(load_max_sq, 1e-9)

    total_metrics['load_variation'] = load_variation
    total_metrics['normalized_load_balance'] = normalized_load_balance
    
    return final_assignments, total_metrics, per_task_results

# ------------------------------------------------------------
# Custom Printer Function for Robust Output
# ------------------------------------------------------------
def print_df_robust(df, title):
    print(f"\n--- {title} ---")
    if not df.empty:
        # Define formatters for numerical columns
        formatters = {col: '{:.4f}'.format for col in df.select_dtypes(include=['float64']).columns}
        
        # Print header
        print(' | '.join([f"{col:<20}" for col in df.columns]))
        print('-' * (20 * len(df.columns) + (len(df.columns) - 1) * 3))
        
        # Print rows
        for index, row in df.iterrows():
            output = []
            for col in df.columns:
                value = row[col]
                if col in formatters:
                    output.append(formatters[col](value).ljust(20))
                elif isinstance(value, (int, np.integer)):
                    output.append(str(value).ljust(20))
                else:
                    output.append(str(value).ljust(20))
            print(' | '.join(output))
    else:
        print("Dataframe is empty.")


# ------------------------------------------------------------
# Demo Execution
# ------------------------------------------------------------
if __name__ == "__main__":
    
    env = QTOEnvMDP(M, N, phi=0.85, gamma=0.95, seed=SEED)
    
    print("####################################################")
    print("# INITIAL QTOP PARAMETERS (SEED 42) #")
    print("####################################################")

    # 1. Print Initial Parameters
    node_params = pd.DataFrame(env.nodes)
    node_params.index.name = 'Node Index'
    print_df_robust(node_params.reset_index(), "Initial Quantum Node Parameters (N=4)")
        
    task_params = pd.DataFrame(env.tasks)
    task_params.index.name = 'Task Index'
    print_df_robust(task_params.reset_index(), "Initial Quantum Task Parameters (M=5)")

    # 2. Run Policy Finding (Value Iteration)
    print("\nStarting Value Iteration to find Optimal Policy...")
    V, policy = value_iteration(env, max_iter=300, tol=1e-6)

    # 3. Print Optimal Policy
    print("\n\n=== Optimal Policy (π(s_{i,j}) from VI) ===")
    for i in range(M):
        for j in range(N):
            s = state_index(i, j, N)
            print(f"s({i},{j}) -> best action (node) = {policy[s]}")
    
    # 4. Run Sequential Simulation
    print("\n\n####################################################")
    print("# EXECUTING QTOP SEQUENTIAL ASSIGNMENT #")
    print("####################################################")
    
    assignments, system_metrics, per_task_data = run_qtop_simulation_from_vi_policy(env, policy)

    # 5. Output Results
    
    print("\n\n####################################################")
    print("# 📊 PER-TASK PERFORMANCE BREAKDOWN 📊 #")
    print("####################################################")

    if per_task_data:
        df_results = pd.DataFrame(per_task_data)
        
        # Rename columns for clarity in the output table
        df_results.columns = [
            'Task Index', 'Assigned Node', 'Queueing Delay (δ_w) [s]', 'Execution Delay (δ_c) [s]', 
            'Total Time (δ_g) [s]', 'Deadline (ϕ_T) [s]', 
            'Energy (e_g^c) [J]', 'Price (P_g) [$]'
        ]
        
        df_results['Status'] = np.where(df_results['Total Time (δ_g) [s]'] < df_results['Deadline (ϕ_T) [s]'], 'OK', 'FAIL')
        
        print_df_robust(df_results, "Per-Task Results")
    else:
        print("No tasks were successfully offloaded due to initial constraint failures.")

    # 6. Print Aggregated System Metrics
    print("\n\n####################################################")
    print("# 📈 AGGREGATED SYSTEM OBJECTIVES 📈 #")
    print("####################################################")
    
    df_agg = pd.DataFrame([system_metrics]).T.reset_index()
    df_agg.columns = ['Metric', 'Value']
    
    print_df_robust(df_agg, "Aggregated System Objectives")

####################################################
# INITIAL QTOP PARAMETERS (SEED 42) #
####################################################

--- Initial Quantum Node Parameters (N=4) ---
Node Index           | qubits               | clops                | power                | price               
----------------------------------------------------------------------------------------------------------------
0.0                  | 8.0                  | 3792.5576            | 0.2361               | 0.2399              
1.0                  | 16.0                 | 1408.8411            | 0.2520               | 0.4231              
2.0                  | 16.0                 | 3414.9041            | 0.4229               | 0.7568              
3.0                  | 4.0                  | 4892.4631            | 0.2514               | 0.5968              

--- Initial Quantum Task Parameters (M=5) ---
Task Index           | qubits               | depth                | shots          

In [4]:
import numpy as np
import random
import pandas as pd

# Set up global parameters for consistency
M = 5  # Number of tasks (m)
N = 4  # Number of nodes (n)
SEED = 42

# ------------------------------------------------------------
# Helper Functions: State Index Mapping
# ------------------------------------------------------------
def state_index(i, j, n_nodes):
    """Map (task i, node j) -> linear state index."""
    return i * n_nodes + j

def state_from_index(s, n_nodes):
    """Inverse mapping from linear index to (i, j)."""
    return s // n_nodes, s % n_nodes


# ------------------------------------------------------------
# Environment / MDP model for QTO (QTOEnvMDP)
# ------------------------------------------------------------
class QTOEnvMDP:
    """
    Models the Quantum Task Offloading (QTO) environment and MDP dynamics.
    """
    def __init__(self, m, n, phi=0.85, gamma=0.95, seed=0):
        
        random.seed(seed)
        np.random.seed(seed)

        self.m = m
        self.n = n
        self.phi = phi
        self.gamma = gamma

        # --- Task parameters (ϕ_q, ϕ_d, ϕ_s, ϕ_T, ϕ_P) ---
        self.tasks = []
        for _ in range(m):
            self.tasks.append({
                "qubits": random.choice([2, 4, 8]),       # ϕ_g^q: required qubits
                "depth": random.uniform(10, 50),          # ϕ_g^d: circuit depth
                "shots": random.choice([100, 500, 1000]), # ϕ_g^s: number of shots
                "deadline": random.uniform(5.0, 30.0),    # ϕ_g^T: execution deadline
                "max_price": random.uniform(5.0, 20.0)    # ϕ_g^P: maximum allowable price
            })

        # --- Node parameters (n_q, n_s, p_j, ρ_j) ---
        self.nodes = []
        for _ in range(n):
            self.nodes.append({
                "qubits": random.choice([4, 8, 16, 32]),  # n_j^q: available qubits
                "clops": random.uniform(1e3, 5e3),        # n_j^s: CLOPS
                "power": random.uniform(0.1, 0.5),        # p_j: power per qubit
                "price": random.uniform(0.1, 1.0)         # ρ_j: price per second
            })

        self.base_node_load = np.zeros(n)  # Used for VI approximation

        # Precompute global maxima for normalization: δ_m, e_m, P_m, σ_m
        self.delta_max, self.energy_max, self.price_max, self.load_max = self._precompute_global_maxima()

    # --------------------------------------------------------
    # Delay model (Eq. 3, 5)
    # --------------------------------------------------------
    def computation_delay(self, task_i, node_k):
        """δ_c_gj = (ϕ_d_g × ϕ_s_g) / n_s_j (Eq. 3)"""
        depth = task_i["depth"]
        shots = task_i["shots"]
        clops = node_k["clops"]
        return (depth * shots) / max(clops, 1e-9) 

    def total_delay(self, i, k):
        """Approximation of total delay for VI (δ_g = δ_c_gj + δ_w_g)."""
        task = self.tasks[i]
        comp = self.computation_delay(task, self.nodes[k])
        # Approximate waiting delay for MDP: (base load + current depth) / CLOPS
        wait = (self.base_node_load[k] + task["depth"]) / max(self.nodes[k]["clops"], 1e-9)
        return comp + wait

    # --------------------------------------------------------
    # Energy, Price, Load, Feasibility, and Transition Logic
    # --------------------------------------------------------

    def energy_consumption(self, i, k):
        """e_c_g = δ_c_gj × ϕ_q_g × p_j (Eq. 9)"""
        task = self.tasks[i]
        node = self.nodes[k]
        δc = self.computation_delay(task, node)
        return δc * task["qubits"] * node["power"]

    def price_consumption(self, i, k):
        """P_g = ρ_j × δ_c_gj (Eq. 14)"""
        task = self.tasks[i]
        node = self.nodes[k]
        δc = self.computation_delay(task, node)
        return node["price"] * δc

    def load_variance_if_assign(self, i, k):
        """Approximation of Var(σ) for MDP reward."""
        loads = self.base_node_load.copy()
        loads[k] += self.tasks[i]["depth"] 
        mean_load = np.mean(loads)
        return np.mean((loads - mean_load) ** 2)

    def P_qubit(self, i, k):
        """Checks qubit constraint (Eq. 24). Returns 1 if feasible, 0 otherwise."""
        req_q = self.tasks[i]["qubits"]
        node_q = self.nodes[k]["qubits"]
        return 1 if node_q >= req_q else 0

    def P_node(self, i, a, k):
        """Transition probability for node selection based on action 'a' and node success ϕ."""
        F = [idx for idx in range(self.n) if self.nodes[idx]["qubits"] >= self.tasks[i]["qubits"]] # Simplified Feasibility check
        if k not in F:
            return 0.0

        if len(F) <= 1:
            return 1.0 if k == a else 0.0

        if k == a:
            return self.phi
        else:
            return (1 - self.phi) / (len(F) - 1)

    def transition_prob(self, s, a, s_next):
        """Full transition probability P(s' | s, a)."""
        i, j = state_from_index(s, self.n)
        i2, k = state_from_index(s_next, self.n)
        if i != i2:
            return 0.0 

        return self.P_node(i, a, k) * self.P_qubit(i, k)

    def normalized_reward(self, i, k, feasible=True,
                          wD=0.4, wE=0.2, wC=0.2, wB=0.2, penalty=100.0):
        """
        Calculates the normalized reward R, aiming to maximize utility (minimize cost).
        """
        if not feasible:
            return -penalty

        D = self.total_delay(i, k)
        E = self.energy_consumption(i, k)
        C = self.price_consumption(i, k)
        B = self.load_variance_if_assign(i, k)

        # Maxima (δ_m, e_m, P_m, σ_m) for normalization
        δm = max(self.delta_max, 1e-9)
        em = max(self.energy_max, 1e-9)
        Pm = max(self.price_max, 1e-9)
        σm = max(self.load_max, 1e-9)

        # Calculate normalized utility (1 - Normalized Cost)
        Dn = 1 - (D / δm)
        En = 1 - (E / em)
        Cn = 1 - (C / Pm)
        Bn = 1 - (B / σm)

        # Ensure normalized metrics are within [0, 1]
        Dn = max(0.0, min(1.0, Dn))
        En = max(0.0, min(1.0, En))
        Cn = max(0.0, min(1.0, Cn))
        Bn = max(0.0, min(1.0, Bn))

        R = wD * Dn + wE * En + wC * Cn + wB * Bn
        return R

    def _precompute_global_maxima(self):
        """Calculates theoretical maximum costs for normalization (δ_m, e_m, P_m, σ_m)."""
        
        # δ_m (Max delay): all tasks on slowest node (min CLOPS) (Eq. 8 approximation)
        slowest_idx = np.argmin([nd["clops"] for nd in self.nodes])
        slow_node = self.nodes[slowest_idx]
        delta_max = 0.0
        for t in self.tasks:
            delta_max += (t["depth"] * t["shots"]) / max(slow_node["clops"], 1e-9)

        # e_m (Max energy): sum of max possible individual energy costs (Eq. 11 approximation)
        energy_max = sum([
            self.computation_delay(t, self.nodes[np.argmin([nd["clops"] for nd in self.nodes])]) * t["qubits"] * self.nodes[np.argmax([nd["power"] for nd in self.nodes])]["power"]
            for t in self.tasks
        ]) 
        
        # P_m (Max price): sum of max possible individual price costs (Eq. 16 approximation)
        price_max = sum([
             self.nodes[np.argmax([nd["price"] for nd in self.nodes])]["price"] * self.computation_delay(t, self.nodes[np.argmin([nd["clops"] for nd in self.nodes])])
            for t in self.tasks
        ])

        # σ_m (Max load): total sum of all task depths (Eq. 13 approximation)
        load_max = sum(t["depth"] for t in self.tasks)

        return delta_max, energy_max, price_max, load_max


# ------------------------------------------------------------
# Value Iteration Algorithm (Policy Finding)
# ------------------------------------------------------------
def value_iteration(env: QTOEnvMDP, max_iter=300, tol=1e-6):
    """
    Computes the optimal Value function V(s) and Policy π(s) for the MDP.
    """
    S = env.m * env.n
    A = env.n

    V = np.zeros(S)
    policy = np.zeros(S, dtype=int)

    for it in range(max_iter):
        delta = 0.0
        V_new = np.copy(V)

        for s in range(S):
            i, j = state_from_index(s, env.n)

            action_values = []
            for a in range(A):
                expected_val = 0.0

                # Sum over all possible next states s_{i,k}
                for k in range(env.n):
                    s_next = state_index(i, k, env.n)

                    P = env.transition_prob(s, a, s_next)
                    if P == 0.0:
                        continue

                    feasible = (env.P_qubit(i, k) == 1)
                    R = env.normalized_reward(i, k, feasible=feasible)

                    expected_val += P * (R + env.gamma * V[s_next])

                action_values.append(expected_val)

            best_val = max(action_values)
            best_action = int(np.argmax(action_values))

            V_new[s] = best_val
            policy[s] = best_action

            delta = max(delta, abs(V_new[s] - V[s]))

        V = V_new

        if delta < tol:
            break

    return V, policy

# ------------------------------------------------------------
# Sequential QTOP Simulation (Policy Execution)
# ------------------------------------------------------------
def run_qtop_simulation_from_vi_policy(env: QTOEnvMDP, policy):
    """
    Simulates the sequential QTOP decision process using the VI policy, 
    enforcing urgency (highest U_g first) and sequential workload accumulation.
    """
    
    # --- 1. Determine Task Urgency Order (U_g) ---
    
    urgency_values = []
    for i in range(env.m):
        task = env.tasks[i]
        feasible = [k for k in range(env.n) if env.nodes[k]["qubits"] >= task["qubits"]]
        
        if not feasible:
            urgency_values.append(np.inf)
            continue

        # Use the fastest feasible node (max CLOPS) for the initial urgency calculation.
        clops_values = [env.nodes[k]["clops"] for k in feasible]
        fastest_node_idx = feasible[np.argmax(clops_values)]
        
        delta_gc = env.computation_delay(task, env.nodes[fastest_node_idx])
        
        # U_g = 1 / (phi_g^T - delta_gj^c) (Eq. 17)
        phi_g_T = task["deadline"]
        urgency = 1.0 / max(1e-9, (phi_g_T - delta_gc))
        urgency_values.append(urgency)

    # Sort tasks by ascending urgency (U_{\pi(1)} <= ...)
    # Then reverse the indices to get DESCENDING (Highest Urgency first)
    task_order_indices = np.argsort(urgency_values)[::-1] 
    
    # --- 2. Sequential Assignment and Workload Update ---
    
    final_assignments = {} 
    current_workload_time = np.zeros(env.n)
    current_workload_load = np.zeros(env.n)
    
    total_metrics = {'delay': 0.0, 'energy': 0.0, 'price': 0.0}
    per_task_results = []
    
    for rank, i in enumerate(task_order_indices): # Iterate over corrected order (Highest U_g first)
        if urgency_values[i] == np.inf:
            continue
            
        task = env.tasks[i]
        
        # Policy lookup
        s_initial = state_index(i, 0, env.n) 
        best_node_k = policy[s_initial] 
        
        # --- Real-Time Metrics Recalculation ---
        
        delta_gc = env.computation_delay(task, env.nodes[best_node_k])
        delta_gw = current_workload_time[best_node_k] # Queueing Delay
        delta_g = delta_gc + delta_gw # Total Time (Eq. 5)
        
        price_g = env.price_consumption(i, best_node_k)
        energy_g = env.energy_consumption(i, best_node_k)
        
        # Check Constraints
        if env.P_qubit(i, best_node_k) == 0: continue
        if delta_g >= task["deadline"]: continue
        if price_g >= task["max_price"]: continue

        # --- Successful Assignment: Update Workload and Metrics ---
        
        current_workload_time[best_node_k] += delta_gc 
        current_workload_load[best_node_k] += task["depth"]
        
        total_metrics['energy'] += energy_g 
        total_metrics['price'] += price_g
        
        final_assignments[i] = best_node_k
        
        # Store per-task results
        per_task_results.append({
            'Task Index': i,
            'Assigned Node': best_node_k,
            'Queueing Delay (δ_w)': delta_gw,
            'Execution Delay (δ_c)': delta_gc,
            'Total Time (δ_g)': delta_g,
            'Deadline (ϕ_T)': task["deadline"],
            'Energy (e_g^c)': energy_g,
            'Price (P_g) [$]': price_g
        })

    # --- 3. Final System Load and Delay Calculation ---
    
    if current_workload_time.size > 0:
        total_metrics['delay'] = np.max(current_workload_time) # δ^t (Eq. 7)
    
    mean_load = np.sum(current_workload_load) / env.n 
    load_variation = np.sum((current_workload_load - mean_load) ** 2) / env.n 
    
    load_max_sq = env.load_max ** 2 
    normalized_load_balance = load_variation / max(load_max_sq, 1e-9)

    total_metrics['load_variation'] = load_variation
    total_metrics['normalized_load_balance'] = normalized_load_balance
    
    return final_assignments, total_metrics, per_task_results

# ------------------------------------------------------------
# Custom Printer Function for Robust Output (avoids external library issues)
# ------------------------------------------------------------
def print_df_robust(df, title):
    print(f"\n--- {title} ---")
    if not df.empty:
        # Define formatters for numerical columns
        formatters = {col: '{:.4f}'.format for col in df.select_dtypes(include=['float64']).columns}
        
        # Print header
        print(' | '.join([f"{col:<20}" for col in df.columns]))
        print('-' * (20 * len(df.columns) + (len(df.columns) - 1) * 3))
        
        # Print rows
        for index, row in df.iterrows():
            output = []
            for col in df.columns:
                value = row[col]
                if col in formatters:
                    output.append(formatters[col](value).ljust(20))
                elif isinstance(value, (int, np.integer)):
                    output.append(str(value).ljust(20))
                else:
                    output.append(str(value).ljust(20))
            print(' | '.join(output))
    else:
        print("Dataframe is empty.")


# ------------------------------------------------------------
# Demo Execution
# ------------------------------------------------------------
if __name__ == "__main__":
    
    env = QTOEnvMDP(M, N, phi=0.85, gamma=0.95, seed=SEED)
    
    print("####################################################")
    print("# INITIAL QTOP PARAMETERS (SEED 42) #")
    print("####################################################")

    # 1. Print Initial Parameters
    node_params = pd.DataFrame(env.nodes)
    node_params.index.name = 'Node Index'
    print_df_robust(node_params.reset_index(), "Initial Quantum Node Parameters (N=4)")
        
    task_params = pd.DataFrame(env.tasks)
    task_params.index.name = 'Task Index'
    print_df_robust(task_params.reset_index(), "Initial Quantum Task Parameters (M=5)")

    # 2. Run Policy Finding (Value Iteration)
    print("\nStarting Value Iteration to find Optimal Policy...")
    V, policy = value_iteration(env, max_iter=300, tol=1e-6)

    # 3. Print Optimal Policy
    print("\n\n=== Optimal Policy (π(s_{i,j}) from VI) ===")
    for i in range(M):
        for j in range(N):
            s = state_index(i, j, N)
            print(f"s({i},{j}) -> best action (node) = {policy[s]}")
    
    # 4. Run Sequential Simulation
    print("\n\n####################################################")
    print("# EXECUTING QTOP SEQUENTIAL ASSIGNMENT #")
    print("####################################################")
    
    assignments, system_metrics, per_task_data = run_qtop_simulation_from_vi_policy(env, policy)

    # 5. Output Results
    
    print("\n\n####################################################")
    print("# 📊 PER-TASK PERFORMANCE BREAKDOWN 📊 #")
    print("####################################################")

    if per_task_data:
        df_results = pd.DataFrame(per_task_data)
        
        # Prepare for robust printing
        df_results.columns = [
            'Task Index', 'Assigned Node', 'Queueing Delay (δ_w)', 'Execution Delay (δ_c)', 
            'Total Time (δ_g)', 'Deadline (ϕ_T)', 
            'Energy (e_g^c) [J]', 'Price (P_g) [$]'
        ]
        
        df_results['Status'] = np.where(df_results['Total Time (δ_g)'] < df_results['Deadline (ϕ_T)'], 'OK', 'FAIL')
        
        print_df_robust(df_results, "Per-Task Results")
    else:
        print("No tasks were successfully offloaded due to initial constraint failures.")

    # 6. Print Aggregated System Metrics
    print("\n\n####################################################")
    print("# 📈 AGGREGATED SYSTEM OBJECTIVES 📈 #")
    print("####################################################")
    
    df_agg = pd.DataFrame(system_metrics.items(), columns=['Metric', 'Value'])
    
    print_df_robust(df_agg, "Aggregated System Objectives")

####################################################
# INITIAL QTOP PARAMETERS (SEED 42) #
####################################################

--- Initial Quantum Node Parameters (N=4) ---
Node Index           | qubits               | clops                | power                | price               
----------------------------------------------------------------------------------------------------------------
0.0                  | 8.0                  | 3792.5576            | 0.2361               | 0.2399              
1.0                  | 16.0                 | 1408.8411            | 0.2520               | 0.4231              
2.0                  | 16.0                 | 3414.9041            | 0.4229               | 0.7568              
3.0                  | 4.0                  | 4892.4631            | 0.2514               | 0.5968              

--- Initial Quantum Task Parameters (M=5) ---
Task Index           | qubits               | depth                | shots          

In [5]:
import numpy as np
import random
import pandas as pd

# Set up global parameters for consistency
M = 5  # Number of tasks (m)
N = 4  # Number of nodes (n)
SEED = 42

# ------------------------------------------------------------
# Helper Functions: State Index Mapping
# ------------------------------------------------------------
def state_index(i, j, n_nodes):
    """Map (task i, node j) -> linear state index."""
    return i * n_nodes + j

def state_from_index(s, n_nodes):
    """Inverse mapping from linear index to (i, j)."""
    return s // n_nodes, s % n_nodes


# ------------------------------------------------------------
# Environment / MDP model for QTO (QTOEnvMDP)
# ------------------------------------------------------------
class QTOEnvMDP:
    """
    Models the Quantum Task Offloading (QTO) environment and MDP dynamics.
    """
    def __init__(self, m, n, phi=0.85, gamma=0.95, seed=0):
        
        random.seed(seed)
        np.random.seed(seed)

        self.m = m
        self.n = n
        self.phi = phi
        self.gamma = gamma

        # --- Task parameters (ϕ_q, ϕ_d, ϕ_s, ϕ_T, ϕ_P) ---
        self.tasks = []
        for _ in range(m):
            self.tasks.append({
                "qubits": random.choice([2, 4, 8]),       # ϕ_g^q: required qubits
                "depth": random.uniform(10, 50),          # ϕ_g^d: circuit depth
                "shots": random.choice([100, 500, 1000]), # ϕ_g^s: number of shots
                "deadline": random.uniform(5.0, 30.0),    # ϕ_g^T: execution deadline
                "max_price": random.uniform(5.0, 20.0)    # ϕ_g^P: maximum allowable price
            })

        # --- Node parameters (n_q, n_s, p_j, ρ_j) ---
        self.nodes = []
        for _ in range(n):
            self.nodes.append({
                "qubits": random.choice([4, 8, 16, 32]),  # n_j^q: available qubits
                "clops": random.uniform(1e3, 5e3),        # n_j^s: CLOPS
                "power": random.uniform(0.1, 0.5),        # p_j: power per qubit
                "price": random.uniform(0.1, 1.0)         # ρ_j: price per second
            })

        self.base_node_load = np.zeros(n)  # Used for VI approximation

        # Precompute global maxima for normalization: δ_m, e_m, P_m, σ_m
        self.delta_max, self.energy_max, self.price_max, self.load_max = self._precompute_global_maxima()

    # --------------------------------------------------------
    # Delay model (Eq. 3, 5)
    # --------------------------------------------------------
    def computation_delay(self, task_i, node_k):
        """δ_c_gj = (ϕ_d_g × ϕ_s_g) / n_s_j (Eq. 3)"""
        depth = task_i["depth"]
        shots = task_i["shots"]
        clops = node_k["clops"]
        return (depth * shots) / max(clops, 1e-9) 

    def total_delay(self, i, k):
        """Approximation of total delay for VI (δ_g = δ_c_gj + δ_w_g)."""
        task = self.tasks[i]
        comp = self.computation_delay(task, self.nodes[k])
        # Approximate waiting delay for MDP: (base load + current depth) / CLOPS
        wait = (self.base_node_load[k] + task["depth"]) / max(self.nodes[k]["clops"], 1e-9)
        return comp + wait

    # --------------------------------------------------------
    # Energy, Price, Load, Feasibility, and Transition Logic
    # --------------------------------------------------------

    def energy_consumption(self, i, k):
        """e_c_g = δ_c_gj × ϕ_q_g × p_j (Eq. 9)"""
        task = self.tasks[i]
        node = self.nodes[k]
        δc = self.computation_delay(task, node)
        return δc * task["qubits"] * node["power"]

    def price_consumption(self, i, k):
        """P_g = ρ_j × δ_c_gj (Eq. 14)"""
        task = self.tasks[i]
        node = self.nodes[k]
        δc = self.computation_delay(task, node)
        return node["price"] * δc

    def load_variance_if_assign(self, i, k):
        """Approximation of Var(σ) for MDP reward."""
        loads = self.base_node_load.copy()
        loads[k] += self.tasks[i]["depth"] 
        mean_load = np.mean(loads)
        return np.mean((loads - mean_load) ** 2)

    def P_qubit(self, i, k):
        """Checks qubit constraint (Eq. 24). Returns 1 if feasible, 0 otherwise."""
        req_q = self.tasks[i]["qubits"]
        node_q = self.nodes[k]["qubits"]
        return 1 if node_q >= req_q else 0

    def P_node(self, i, a, k):
        """Transition probability for node selection based on action 'a' and node success ϕ."""
        F = [idx for idx in range(self.n) if self.nodes[idx]["qubits"] >= self.tasks[i]["qubits"]] # Simplified Feasibility check
        if k not in F:
            return 0.0

        if len(F) <= 1:
            return 1.0 if k == a else 0.0

        if k == a:
            return self.phi
        else:
            return (1 - self.phi) / (len(F) - 1)

    def transition_prob(self, s, a, s_next):
        """Full transition probability P(s' | s, a)."""
        i, j = state_from_index(s, self.n)
        i2, k = state_from_index(s_next, self.n)
        if i != i2:
            return 0.0 

        return self.P_node(i, a, k) * self.P_qubit(i, k)

    def normalized_reward(self, i, k, feasible=True,
                          wD=0.4, wE=0.2, wC=0.2, wB=0.2, penalty=100.0):
        """
        Calculates the normalized reward R, aiming to maximize utility (minimize cost).
        """
        if not feasible:
            return -penalty

        D = self.total_delay(i, k)
        E = self.energy_consumption(i, k)
        C = self.price_consumption(i, k)
        B = self.load_variance_if_assign(i, k)

        # Maxima (δ_m, e_m, P_m, σ_m) for normalization
        δm = max(self.delta_max, 1e-9)
        em = max(self.energy_max, 1e-9)
        Pm = max(self.price_max, 1e-9)
        σm = max(self.load_max, 1e-9)

        # Calculate normalized utility (1 - Normalized Cost)
        Dn = 1 - (D / δm)
        En = 1 - (E / em)
        Cn = 1 - (C / Pm)
        Bn = 1 - (B / σm)

        # Ensure normalized metrics are within [0, 1]
        Dn = max(0.0, min(1.0, Dn))
        En = max(0.0, min(1.0, En))
        Cn = max(0.0, min(1.0, Cn))
        Bn = max(0.0, min(1.0, Bn))

        R = wD * Dn + wE * En + wC * Cn + wB * Bn
        return R

    def _precompute_global_maxima(self):
        """Calculates theoretical maximum costs for normalization (δ_m, e_m, P_m, σ_m)."""
        
        # δ_m (Max delay): all tasks on slowest node (min CLOPS) (Eq. 8 approximation)
        slowest_idx = np.argmin([nd["clops"] for nd in self.nodes])
        slow_node = self.nodes[slowest_idx]
        delta_max = 0.0
        for t in self.tasks:
            delta_max += (t["depth"] * t["shots"]) / max(slow_node["clops"], 1e-9)

        # e_m (Max energy): sum of max possible individual energy costs (Eq. 11 approximation)
        energy_max = sum([
            self.computation_delay(t, self.nodes[np.argmin([nd["clops"] for nd in self.nodes])]) * t["qubits"] * self.nodes[np.argmax([nd["power"] for nd in self.nodes])]["power"]
            for t in self.tasks
        ]) 
        
        # P_m (Max price): sum of max possible individual price costs (Eq. 16 approximation)
        price_max = sum([
             self.nodes[np.argmax([nd["price"] for nd in self.nodes])]["price"] * self.computation_delay(t, self.nodes[np.argmin([nd["clops"] for nd in self.nodes])])
            for t in self.tasks
        ])

        # σ_m (Max load): total sum of all task depths (Eq. 13 approximation)
        load_max = sum(t["depth"] for t in self.tasks)

        return delta_max, energy_max, price_max, load_max


# ------------------------------------------------------------
# Value Iteration Algorithm (Policy Finding)
# ------------------------------------------------------------
def value_iteration(env: QTOEnvMDP, max_iter=300, tol=1e-6):
    """
    Computes the optimal Value function V(s) and Policy π(s) for the MDP.
    """
    S = env.m * env.n
    A = env.n

    V = np.zeros(S)
    policy = np.zeros(S, dtype=int)

    for it in range(max_iter):
        delta = 0.0
        V_new = np.copy(V)

        for s in range(S):
            i, j = state_from_index(s, env.n)

            action_values = []
            for a in range(A):
                expected_val = 0.0

                # Sum over all possible next states s_{i,k}
                for k in range(env.n):
                    s_next = state_index(i, k, env.n)

                    P = env.transition_prob(s, a, s_next)
                    if P == 0.0:
                        continue

                    feasible = (env.P_qubit(i, k) == 1)
                    R = env.normalized_reward(i, k, feasible=feasible)

                    expected_val += P * (R + env.gamma * V[s_next])

                action_values.append(expected_val)

            best_val = max(action_values)
            best_action = int(np.argmax(action_values))

            V_new[s] = best_val
            policy[s] = best_action

            delta = max(delta, abs(V_new[s] - V[s]))

        V = V_new

        if delta < tol:
            break

    return V, policy

# ------------------------------------------------------------
# Sequential QTOP Simulation (Policy Execution)
# ------------------------------------------------------------
def run_qtop_simulation_from_vi_policy(env: QTOEnvMDP, policy):
    """
    Simulates the sequential QTOP decision process using the VI policy, 
    enforcing urgency (highest U_g first) and sequential workload accumulation.
    """
    
    # --- 1. Determine Task Urgency Order (U_g) ---
    
    urgency_values = []
    for i in range(env.m):
        task = env.tasks[i]
        feasible = [k for k in range(env.n) if env.nodes[k]["qubits"] >= task["qubits"]]
        
        if not feasible:
            urgency_values.append(np.inf)
            continue

        # Use the fastest feasible node (max CLOPS) for the initial urgency calculation.
        clops_values = [env.nodes[k]["clops"] for k in feasible]
        fastest_node_idx = feasible[np.argmax(clops_values)]
        
        delta_gc = env.computation_delay(task, env.nodes[fastest_node_idx])
        
        # U_g = 1 / (phi_g^T - delta_gj^c) (Eq. 17)
        phi_g_T = task["deadline"]
        urgency = 1.0 / max(1e-9, (phi_g_T - delta_gc))
        urgency_values.append(urgency)

    # Sort tasks by ascending urgency (U_{\pi(1)} <= ...)
    # Then reverse the indices to get DESCENDING (Highest Urgency first)
    task_order_indices = np.argsort(urgency_values)[::-1] 
    
    # --- 2. Sequential Assignment and Workload Update ---
    
    final_assignments = {} 
    current_workload_time = np.zeros(env.n)
    current_workload_load = np.zeros(env.n)
    
    total_metrics = {'delay': 0.0, 'energy': 0.0, 'price': 0.0}
    per_task_results = []
    
    for rank, i in enumerate(task_order_indices): # Iterate over corrected order (Highest U_g first)
        if urgency_values[i] == np.inf:
            continue
            
        task = env.tasks[i]
        
        # Policy lookup
        s_initial = state_index(i, 0, env.n) 
        best_node_k = policy[s_initial] 
        
        # --- Real-Time Metrics Recalculation ---
        
        delta_gc = env.computation_delay(task, env.nodes[best_node_k])
        delta_gw = current_workload_time[best_node_k] # Queueing Delay
        delta_g = delta_gc + delta_gw # Total Time (Eq. 5)
        
        price_g = env.price_consumption(i, best_node_k)
        energy_g = env.energy_consumption(i, best_node_k)
        
        # Check Constraints
        if env.P_qubit(i, best_node_k) == 0: continue
        if delta_g >= task["deadline"]: continue
        if price_g >= task["max_price"]: continue

        # --- Successful Assignment: Update Workload and Metrics ---
        
        current_workload_time[best_node_k] += delta_gc # Update δ_j
        current_workload_load[best_node_k] += task["depth"] # Update σ_j
        
        total_metrics['energy'] += energy_g 
        total_metrics['price'] += price_g
        
        final_assignments[i] = best_node_k
        
        # Store per-task results
        per_task_results.append({
            'Task Index': i,
            'Assigned Node': best_node_k,
            'Queueing Delay (δ_w)': delta_gw,
            'Execution Delay (δ_c)': delta_gc,
            'Total Time (δ_g)': delta_g,
            'Deadline (ϕ_T)': task["deadline"],
            'Energy (e_g^c) [J]': energy_g,
            'Price (P_g) [$]': price_g
        })

    # --- 3. Final System Load and Delay Calculation ---
    
    if current_workload_time.size > 0:
        total_metrics['delay'] = np.max(current_workload_time) # δ^t (Eq. 7)
    
    mean_load = np.sum(current_workload_load) / env.n 
    load_variation = np.sum((current_workload_load - mean_load) ** 2) / env.n 
    
    load_max_sq = env.load_max ** 2 
    normalized_load_balance = load_variation / max(load_max_sq, 1e-9)

    total_metrics['load_variation'] = load_variation
    total_metrics['normalized_load_balance'] = normalized_load_balance
    
    return final_assignments, total_metrics, per_task_results

# ------------------------------------------------------------
# Custom Printer Function for Robust Output (avoids external library issues)
# ------------------------------------------------------------
def print_df_robust(df, title):
    print(f"\n--- {title} ---")
    if not df.empty:
        # Define formatters for numerical columns
        formatters = {col: '{:.4f}'.format for col in df.select_dtypes(include=['float64']).columns}
        
        # Print header
        print(' | '.join([f"{col:<20}" for col in df.columns]))
        print('-' * (20 * len(df.columns) + (len(df.columns) - 1) * 3))
        
        # Print rows
        for index, row in df.iterrows():
            output = []
            for col in df.columns:
                value = row[col]
                if col in formatters:
                    output.append(formatters[col](value).ljust(20))
                elif isinstance(value, (int, np.integer)):
                    output.append(str(value).ljust(20))
                else:
                    output.append(str(value).ljust(20))
            print(' | '.join(output))
    else:
        print("Dataframe is empty.")


# ------------------------------------------------------------
# Demo Execution
# ------------------------------------------------------------
if __name__ == "__main__":
    
    env = QTOEnvMDP(M, N, phi=0.85, gamma=0.95, seed=SEED)
    
    print("####################################################")
    print("# INITIAL QTOP PARAMETERS (SEED 42) #")
    print("####################################################")

    # 1. Print Initial Parameters
    node_params = pd.DataFrame(env.nodes)
    node_params.index.name = 'Node Index'
    print_df_robust(node_params.reset_index(), "Initial Quantum Node Parameters (N=4)")
        
    task_params = pd.DataFrame(env.tasks)
    task_params.index.name = 'Task Index'
    print_df_robust(task_params.reset_index(), "Initial Quantum Task Parameters (M=5)")

    # 2. Run Policy Finding (Value Iteration)
    print("\nStarting Value Iteration to find Optimal Policy...")
    V, policy = value_iteration(env, max_iter=300, tol=1e-6)

    # 3. Print Optimal Policy
    print("\n\n=== Optimal Policy (π(s_{i,j}) from VI) ===")
    for i in range(M):
        for j in range(N):
            s = state_index(i, j, N)
            print(f"s({i},{j}) -> best action (node) = {policy[s]}")
    
    # 4. Run Sequential Simulation
    print("\n\n####################################################")
    print("# EXECUTING QTOP SEQUENTIAL ASSIGNMENT #")
    print("####################################################")
    
    assignments, system_metrics, per_task_data = run_qtop_simulation_from_vi_policy(env, policy)

    # 5. Output Results
    
    print("\n\n####################################################")
    print("# 📊 PER-TASK PERFORMANCE BREAKDOWN 📊 #")
    print("####################################################")

    if per_task_data:
        df_results = pd.DataFrame(per_task_data)
        
        # Prepare for robust printing
        df_results.columns = [
            'Task Index', 'Assigned Node', 'Queueing Delay (δ_w)', 'Execution Delay (δ_c)', 
            'Total Time (δ_g)', 'Deadline (ϕ_T)', 
            'Energy (e_g^c) [J]', 'Price (P_g) [$]'
        ]
        
        df_results['Status'] = np.where(df_results['Total Time (δ_g)'] < df_results['Deadline (ϕ_T)'], 'OK', 'FAIL')
        
        print_df_robust(df_results, "Per-Task Results")
    else:
        print("No tasks were successfully offloaded due to initial constraint failures.")

    # 6. Print Aggregated System Metrics
    print("\n\n####################################################")
    print("# 📈 AGGREGATED SYSTEM OBJECTIVES 📈 #")
    print("####################################################")
    
    df_agg = pd.DataFrame(system_metrics.items(), columns=['Metric', 'Value'])
    
    print_df_robust(df_agg, "Aggregated System Objectives")

####################################################
# INITIAL QTOP PARAMETERS (SEED 42) #
####################################################

--- Initial Quantum Node Parameters (N=4) ---
Node Index           | qubits               | clops                | power                | price               
----------------------------------------------------------------------------------------------------------------
0.0                  | 8.0                  | 3792.5576            | 0.2361               | 0.2399              
1.0                  | 16.0                 | 1408.8411            | 0.2520               | 0.4231              
2.0                  | 16.0                 | 3414.9041            | 0.4229               | 0.7568              
3.0                  | 4.0                  | 4892.4631            | 0.2514               | 0.5968              

--- Initial Quantum Task Parameters (M=5) ---
Task Index           | qubits               | depth                | shots          